# Emotive TTS — Colab Pipeline

**COMP3065 Project** — Emotion-Conditioned Neural TTS via VITS Fine-Tuning

| Section | Task | Systems |
|---------|------|---------|
| 0 | Setup — Mount Drive, clone repo, install deps | — |
| 1 | Data preparation — Download & process EmoV-DB | — |
| 2 | Train System A — Domain adaptation (no emotion) | A |
| 3 | Train System B — + emotion embedding | B |
| 4 | Train System C — + prosody heads | C |
| 5 | Inference — Generate eval stimuli (A0/A/B/C) | All |
| 6 | Evaluation — Prosody analysis + SER probe | All |
| 7 | Visualization — System comparison plots & analysis | All |

**Runtime:** GPU → T4 (Runtime → Change runtime type → T4 GPU)
**GitHub:** https://github.com/jimmy00415/COMP3067_Project

---

> **Resumability:** Every stage saves checkpoints to Google Drive.
> If the session disconnects, **Runtime → Run all** — completed stages are skipped automatically.

### System Architecture (Causal Chain)
```
A0 (pretrained VITS) → A (domain-adapted) → B (+ emotion embedding) → C (+ prosody heads)
```
Each system initializes from the previous checkpoint — clean attribution of each improvement.

## 0. Setup

In [1]:
# ══ 0a. Mount Google Drive ══
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PROJECT = '/content/drive/MyDrive/COMP3065_Project'
for subdir in ['checkpoints/system_a', 'checkpoints/system_b',
               'checkpoints/system_c', 'data/processed', 'data/raw',
               'data/manifests', 'tables', 'figures', 'outputs']:
    os.makedirs(os.path.join(DRIVE_PROJECT, subdir), exist_ok=True)

print(f'Drive project folder: {DRIVE_PROJECT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project folder: /content/drive/MyDrive/COMP3065_Project


In [2]:
# ══ 0b. Clone / pull project from GitHub ══
import os

REPO_URL = 'https://github.com/jimmy00415/COMP3067_Project.git'
LOCAL_PROJECT = '/content/COMP3067_Project'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    print('Repo exists — pulling latest...')
    !cd {LOCAL_PROJECT} && git pull --ff-only
else:
    if os.path.exists(LOCAL_PROJECT):
        import shutil
        shutil.rmtree(LOCAL_PROJECT)
    !git clone {REPO_URL} {LOCAL_PROJECT}

%cd {LOCAL_PROJECT}
print(f'Working directory: {os.getcwd()}')

Repo exists — pulling latest...
Already up to date.
/content/COMP3067_Project
Working directory: /content/COMP3067_Project


In [3]:
# ══ 0c. Install dependencies ══
#
# Strategy:
#   1. Check if TTS is already importable (post-restart → skip install)
#   2. If not: install espeak-ng, coqui-tts, pin numpy<2, restart runtime
#   3. After restart: Runtime → Run all continues from here and skips
#
import sys, os

def _deps_ok():
    """Return True if all critical packages load without C-ABI crash."""
    try:
        import numpy
        numpy.array([1.0]).astype(numpy.float32)   # C-ABI smoke test
        import TTS
        import librosa
        import soundfile
        import pandas
        import scipy
        import sklearn
        import seaborn
        return True
    except Exception:
        return False

if _deps_ok():
    import TTS, numpy
    print(f'✅ All deps OK — TTS {TTS.__version__}, numpy {numpy.__version__}')
else:
    print(f'Python {sys.version}')

    # 1. System dep: espeak-ng (survives restarts)
    print('[1/4] Installing espeak-ng ...')
    !apt-get install -y -qq espeak-ng > /dev/null 2>&1

    # 2. Pin numpy<2 BEFORE coqui-tts (prevents 2.x from being pulled)
    print('[2/4] Installing numpy<2 + coqui-tts ...')
    !pip install -q 'numpy>=1.24,<2.0'
    !pip install -q coqui-tts 'numpy>=1.24,<2.0'
    # Re-pin numpy in case coqui-tts pulled 2.x transitively
    !pip install -q 'numpy>=1.24,<2.0'

    # 3. Remaining deps
    print('[3/4] Installing remaining deps ...')
    !pip install -q 'speechbrain>=1.0' 'omegaconf>=2.3' 'mlflow>=2.10'
    !pip install -q 'pandas>=2.0' 'matplotlib>=3.7' 'seaborn>=0.13'
    !pip install -q 'scipy>=1.11' 'scikit-learn>=1.3' 'soundfile>=0.12'
    !pip install -q 'librosa>=0.10'

    # 4. Install project in editable mode
    !pip install -q -e .

    # Final numpy pin (belt-and-suspenders)
    !pip install -q 'numpy>=1.24,<2.0'

    print()
    print('[4/4] Restarting runtime to load numpy C extensions cleanly ...')
    print('      After restart, click  Runtime → Run all  to continue.')
    os.kill(os.getpid(), 9)  # hard restart

✅ All deps OK — TTS 0.27.5, numpy 1.26.4


In [4]:
# ══ 0d. Post-restart re-init ══
#
# After runtime restart ALL Python state is lost.
# This cell re-establishes every variable the rest of the notebook needs.
#
import os, sys, shutil, logging, gc, yaml

# ---------- constants (must match 0a / 0b) ----------
DRIVE_PROJECT  = '/content/drive/MyDrive/COMP3065_Project'
LOCAL_PROJECT  = '/content/COMP3067_Project'

# Mount Drive if not already mounted (happens after restart)
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Make sure we are in the repo directory
os.chdir(LOCAL_PROJECT)
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

# Install project package if needed (editable mode survives restart)
try:
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS
except ImportError:
    os.system(f'{sys.executable} -m pip install -q -e .')
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS

# Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s',
                    force=True)

# Verify
import torch, numpy as np
print(f'PyTorch  : {torch.__version__}')
print(f'numpy    : {np.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU      : {gpu}  ({vram:.1f} GB)')
else:
    print('⚠️  WARNING: No GPU detected — training will be extremely slow.')
print(f'Emotions : {EMOTION_LABELS}')
print(f'CWD      : {os.getcwd()}')

# Load data config (used throughout the notebook)
with open('configs/data.yaml', 'r') as f:
    DATA_CFG = yaml.safe_load(f)
print(f'Data cfg : loaded ({DATA_CFG["dataset"]["name"]})')

PyTorch  : 2.10.0+cu128
numpy    : 1.26.4
CUDA     : True
GPU      : Tesla T4  (15.6 GB)
Emotions : ['neutral', 'angry', 'amused', 'disgust']
CWD      : /content/COMP3067_Project
Data cfg : loaded (emov_db)


## 1. Data Preparation

In [5]:
# ══ 1a. Download EmoV-DB from OpenSLR ══
import os, glob, tarfile, shutil

EMOVDB_DRIVE = os.path.join(DRIVE_PROJECT, 'data', 'raw', 'EmoV-DB')
EMOVDB_LOCAL = 'data/raw/EmoV-DB'
os.makedirs('data/raw', exist_ok=True)

def _count_wavs(d):
    if not os.path.isdir(d):
        return 0
    return len(glob.glob(os.path.join(d, '**', '*.wav'), recursive=True))

MIN_WAVS = 100  # EmoV-DB has ~6000+ wavs

# Priority 1: already local with enough files
if os.path.isdir(EMOVDB_LOCAL) and _count_wavs(EMOVDB_LOCAL) >= MIN_WAVS:
    print(f'✅ EmoV-DB found locally ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Priority 2: Drive cache
elif os.path.isdir(EMOVDB_DRIVE) and _count_wavs(EMOVDB_DRIVE) >= MIN_WAVS:
    if not os.path.exists(EMOVDB_LOCAL):
        os.symlink(EMOVDB_DRIVE, EMOVDB_LOCAL)
    print(f'✅ EmoV-DB symlinked from Drive ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Priority 3: download from OpenSLR
else:
    print('Downloading EmoV-DB from OpenSLR (may take 10-15 min) ...')
    os.makedirs(EMOVDB_LOCAL, exist_ok=True)

    BASE_URL = 'https://www.openslr.org/resources/115'
    speakers  = ['bea', 'jenie', 'josh', 'sam']
    emotions  = {'Amused': 'Amused', 'Angry': 'Angry',
                 'Disgusted': 'Disgust', 'Neutral': 'Neutral'}

    for emo_slr, emo_folder in emotions.items():
        emo_dir = os.path.join(EMOVDB_LOCAL, emo_folder)
        os.makedirs(emo_dir, exist_ok=True)
        for spk in speakers:
            fname    = f'{spk}_{emo_slr}.tar.gz'
            tar_path = f'/tmp/{fname}'
            if not os.path.isfile(tar_path) or os.path.getsize(tar_path) < 10_000:
                ret = os.system(f'wget -q --show-progress -O {tar_path} {BASE_URL}/{fname}')
                if ret != 0 or not os.path.isfile(tar_path) or os.path.getsize(tar_path) < 10_000:
                    print(f'  ⚠ skip {fname} (not available)')
                    if os.path.isfile(tar_path):
                        os.remove(tar_path)
                    continue
            try:
                with tarfile.open(tar_path, 'r:gz') as tf:
                    tf.extractall(emo_dir)
                print(f'  ✓ extracted {fname} → {emo_folder}/')
            except Exception as e:
                print(f'  ⚠ WARN: {fname}: {e}')

    # Persist to Drive for next session
    print('Persisting to Drive ...')
    if not os.path.isdir(EMOVDB_DRIVE):
        os.makedirs(os.path.dirname(EMOVDB_DRIVE), exist_ok=True)
        shutil.copytree(EMOVDB_LOCAL, EMOVDB_DRIVE)
    print(f'✅ EmoV-DB ready ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Quick audit — show files per emotion
print('\n📊 EmoV-DB file audit:')
for d in sorted(os.listdir(EMOVDB_LOCAL)):
    p = os.path.join(EMOVDB_LOCAL, d)
    if os.path.isdir(p):
        print(f'  {d:12s}: {_count_wavs(p):>5d} wav files')

✅ EmoV-DB symlinked from Drive (3108 wavs)

📊 EmoV-DB file audit:
  Amused      :   778 wav files
  Angry       :   640 wav files
  Disgust     :   798 wav files
  Neutral     :   892 wav files


In [6]:
# ══ 1b. Run data preparation pipeline ══
import os, yaml, shutil, glob
from pathlib import Path

MANIFESTS_DIR = 'data/manifests'
PROCESSED_DIR = 'data/processed/train'
train_csv = os.path.join(MANIFESTS_DIR, 'train.csv')

# Check Drive for existing manifests first
drive_train_csv = os.path.join(DRIVE_PROJECT, 'data', 'manifests', 'train.csv')
if not os.path.isfile(train_csv) and os.path.isfile(drive_train_csv):
    os.makedirs(MANIFESTS_DIR, exist_ok=True)
    drive_manifests = os.path.join(DRIVE_PROJECT, 'data', 'manifests')
    for f in Path(drive_manifests).glob('*.csv'):
        shutil.copy2(str(f), MANIFESTS_DIR)
    print('Restored manifests from Drive.')

# Also restore processed audio from Drive if available
drive_processed = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'train')
if not os.path.isdir(PROCESSED_DIR) or len(glob.glob(os.path.join(PROCESSED_DIR, '**/*.wav'), recursive=True)) < 10:
    if os.path.isdir(drive_processed) and len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True)) > 100:
        print('Restoring processed audio from Drive ...')
        os.makedirs(PROCESSED_DIR, exist_ok=True)
        shutil.copytree(drive_processed, PROCESSED_DIR, dirs_exist_ok=True)
        n_restored = len(glob.glob(os.path.join(PROCESSED_DIR, '**/*.wav'), recursive=True))
        print(f'Restored {n_restored} processed audio files from Drive.')

# Validate that processed audio actually exists (not just the CSV)
_audio_ok = False
if os.path.isfile(train_csv):
    import pandas as pd
    df = pd.read_csv(train_csv)
    _pcol = 'processed_path' if 'processed_path' in df.columns else 'file_path'
    if _pcol in df.columns:
        _n_exist = df[_pcol].head(20).apply(lambda p: os.path.isfile(str(p))).sum()
        _audio_ok = _n_exist > 0
    if _audio_ok:
        print(f'✅ Manifests + audio OK ({len(df)} samples) — skipping data prep.')
        print(df['emotion'].value_counts().to_string())
    else:
        print('Manifest exists but processed audio files missing — re-running data prep.')

if not _audio_ok:
    from src.data.prepare import prepare_dataset, scan_emovdb, audit_speaker_coverage

    with open('configs/data.yaml', 'r') as f:
        data_cfg = yaml.safe_load(f)

    data_cfg['dataset']['raw_dir'] = EMOVDB_LOCAL

    # --- Speaker Audit (display before processing) ---
    print('\n📊 Speaker Audit:')
    raw_df = scan_emovdb(EMOVDB_LOCAL)
    coverage = audit_speaker_coverage(raw_df)
    from IPython.display import display
    display(coverage)
    print()

    # --- Run full data preparation pipeline ---
    summary = prepare_dataset(data_cfg)

    print('\n✅ Dataset Summary:')
    for k, v in summary.items():
        print(f'  {k}: {v}')

    # Persist manifests to Drive
    drive_manifests = os.path.join(DRIVE_PROJECT, 'data', 'manifests')
    os.makedirs(drive_manifests, exist_ok=True)
    for f in Path(MANIFESTS_DIR).glob('*.csv'):
        shutil.copy2(str(f), drive_manifests)
    print('Manifests saved to Drive.')

    # Persist processed audio to Drive for future sessions
    if os.path.isdir(PROCESSED_DIR):
        if not os.path.isdir(drive_processed) or len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True)) < 100:
            print('Saving processed audio to Drive (first time only) ...')
            os.makedirs(drive_processed, exist_ok=True)
            shutil.copytree(PROCESSED_DIR, drive_processed, dirs_exist_ok=True)
            n_saved = len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True))
            print(f'Saved {n_saved} processed audio files to Drive.')

Restored manifests from Drive.
Restoring processed audio from Drive ...
Restored 3108 processed audio files from Drive.
✅ Manifests + audio OK (2486 samples) — skipping data prep.
emotion
neutral    714
disgust    638
amused     622
angry      512


## 2. Training: System A — Domain Adaptation

Fine-tunes pretrained VITS on EmoV-DB **without** emotion labels.
This is the domain-adapted baseline that isolates the conditioning effect (A → B).

In [ ]:
# ══ 1c. Checkpoint Reset Control ══
#
# ⚠️  Set FORCE_RETRAIN = True to delete ALL existing checkpoints and
#     retrain from scratch. This is needed after code changes that affect
#     the training loop (e.g. loss function fixes).
#
#     After a clean run completes, set it back to False.
#
import os, shutil

FORCE_RETRAIN = False  # ← Set to True to force retraining

if FORCE_RETRAIN:
    print('⚠️  FORCE_RETRAIN is ON — deleting all checkpoints ...')
    for sys_name in ['system_a', 'system_b', 'system_c']:
        for base in ['.', DRIVE_PROJECT]:
            ckpt = os.path.join(base, 'checkpoints', sys_name, 'best.pth')
            if os.path.isfile(ckpt):
                os.remove(ckpt)
                print(f'  ✗ Deleted {ckpt}')
    # Also clear stale eval stimuli (generated from old checkpoints)
    for base in ['.', DRIVE_PROJECT]:
        eval_dir = os.path.join(base, 'data', 'processed', 'eval_stimuli')
        if os.path.isdir(eval_dir):
            shutil.rmtree(eval_dir)
            print(f'  ✗ Deleted {eval_dir}')
    print('✅ All checkpoints cleared — training will run fresh.')
else:
    print('ℹ️  FORCE_RETRAIN = False — existing checkpoints will be reused.')

In [ ]:
# ══ 2a. Train System A ══
import os, yaml, torch, shutil, gc

CKPT_A_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_a', 'best.pth')
CKPT_A_LOCAL = 'checkpoints/system_a/best.pth'
os.makedirs(os.path.dirname(CKPT_A_LOCAL), exist_ok=True)

# Restore from Drive if available
if os.path.isfile(CKPT_A_DRIVE) and not os.path.isfile(CKPT_A_LOCAL):
    shutil.copy2(CKPT_A_DRIVE, CKPT_A_LOCAL)
    print('System A checkpoint restored from Drive.')

if os.path.isfile(CKPT_A_LOCAL):
    sz = os.path.getsize(CKPT_A_LOCAL) / 1e6
    print(f'✅ System A checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_a.yaml', 'r') as f:
        cfg_a = yaml.safe_load(f)

    # Colab overrides
    cfg_a['use_cuda'] = torch.cuda.is_available()
    cfg_a.setdefault('training', {})
    cfg_a['training']['fp16'] = torch.cuda.is_available()
    cfg_a['training']['num_workers'] = 0  # Colab stability
    cfg_a['training']['init_from'] = 'pretrained'
    cfg_a['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_A_LOCAL)

    # Increase batch size if VRAM allows (T4 = 15 GB)
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb >= 14:
            cfg_a['training']['batch_size'] = 32
            print(f'VRAM {vram_gb:.1f} GB — using batch_size=32 for speed')
        else:
            cfg_a['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    print('🚀 Starting System A training (domain adaptation, no emotion labels) ...')
    print(f'   Epochs: {cfg_a["training"].get("max_epochs", 20)}, '
          f'LR: {cfg_a["training"]["optimizer"]["lr"]}, '
          f'Batch: {cfg_a["training"].get("batch_size", 16)}, '
          f'Val every: {cfg_a["training"].get("val_every", 5)} epochs')

    history_a = train(cfg_a)

    # Backup to Drive
    if os.path.isfile(CKPT_A_LOCAL):
        os.makedirs(os.path.dirname(CKPT_A_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_A_LOCAL, CKPT_A_DRIVE)
        print('✅ System A checkpoint saved to Drive.')
    else:
        print('⚠️  No best.pth found — training may have been too short.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

System A checkpoint restored from Drive.
✅ System A checkpoint exists (202.5 MB) — SKIPPING training.


## 3. Training: System B — Emotion Embedding

System B = System A + `nn.Embedding(4, 192)` for emotion conditioning.
Warm-starts from the System A checkpoint.

In [ ]:
# ══ 3a. Train System B ══
import os, yaml, torch, shutil, gc

CKPT_B_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_b', 'best.pth')
CKPT_B_LOCAL = 'checkpoints/system_b/best.pth'
os.makedirs(os.path.dirname(CKPT_B_LOCAL), exist_ok=True)

if os.path.isfile(CKPT_B_DRIVE) and not os.path.isfile(CKPT_B_LOCAL):
    shutil.copy2(CKPT_B_DRIVE, CKPT_B_LOCAL)
    print('System B checkpoint restored from Drive.')

if os.path.isfile(CKPT_B_LOCAL):
    sz = os.path.getsize(CKPT_B_LOCAL) / 1e6
    print(f'✅ System B checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_b.yaml', 'r') as f:
        cfg_b = yaml.safe_load(f)

    cfg_b['use_cuda'] = torch.cuda.is_available()
    cfg_b.setdefault('training', {})
    cfg_b['training']['fp16'] = torch.cuda.is_available()
    cfg_b['training']['num_workers'] = 0
    cfg_b['training']['init_from'] = os.path.abspath(CKPT_A_LOCAL)
    cfg_b['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_B_LOCAL)

    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb >= 14:
            cfg_b['training']['batch_size'] = 32
            print(f'VRAM {vram_gb:.1f} GB — using batch_size=32 for speed')
        else:
            cfg_b['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    print('🚀 Starting System B training (+ emotion embedding) ...')
    print(f'   Init from: System A | Epochs: {cfg_b["training"].get("max_epochs", 20)}, '
          f'LR: {cfg_b["training"]["optimizer"]["lr"]}, '
          f'Batch: {cfg_b["training"].get("batch_size", 16)}, '
          f'Val every: {cfg_b["training"].get("val_every", 5)} epochs')

    history_b = train(cfg_b)

    if os.path.isfile(CKPT_B_LOCAL):
        os.makedirs(os.path.dirname(CKPT_B_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_B_LOCAL, CKPT_B_DRIVE)
        print('✅ System B checkpoint saved to Drive.')
    else:
        print('⚠️  No best.pth found — training may have been too short.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

System B checkpoint restored from Drive.
✅ System B checkpoint exists (202.5 MB) — SKIPPING training.


## 4. Training: System C — Prosody Supervision

System C = System B + utterance-level prosody auxiliary heads (F0 + energy).
Warm-starts from the System B checkpoint. Loss weight λ = 0.1.

In [ ]:
# ══ 4a. Train System C ══
import os, yaml, torch, shutil, gc

CKPT_C_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_c', 'best.pth')
CKPT_C_LOCAL = 'checkpoints/system_c/best.pth'
os.makedirs(os.path.dirname(CKPT_C_LOCAL), exist_ok=True)

if os.path.isfile(CKPT_C_DRIVE) and not os.path.isfile(CKPT_C_LOCAL):
    shutil.copy2(CKPT_C_DRIVE, CKPT_C_LOCAL)
    print('System C checkpoint restored from Drive.')

if os.path.isfile(CKPT_C_LOCAL):
    sz = os.path.getsize(CKPT_C_LOCAL) / 1e6
    print(f'✅ System C checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_c.yaml', 'r') as f:
        cfg_c = yaml.safe_load(f)

    cfg_c['use_cuda'] = torch.cuda.is_available()
    cfg_c.setdefault('training', {})
    cfg_c['training']['fp16'] = torch.cuda.is_available()
    cfg_c['training']['num_workers'] = 0
    cfg_c['training']['init_from'] = os.path.abspath(CKPT_B_LOCAL)
    cfg_c['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_C_LOCAL)

    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb >= 14:
            cfg_c['training']['batch_size'] = 32
            print(f'VRAM {vram_gb:.1f} GB — using batch_size=32 for speed')
        else:
            cfg_c['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    print('🚀 Starting System C training (+ prosody heads, λ=0.1) ...')
    print(f'   Init from: System B | Epochs: {cfg_c["training"].get("max_epochs", 20)}, '
          f'LR: {cfg_c["training"]["optimizer"]["lr"]}, '
          f'Batch: {cfg_c["training"].get("batch_size", 16)}, '
          f'Val every: {cfg_c["training"].get("val_every", 5)} epochs')

    history_c = train(cfg_c)

    if os.path.isfile(CKPT_C_LOCAL):
        os.makedirs(os.path.dirname(CKPT_C_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_C_LOCAL, CKPT_C_DRIVE)
        print('✅ System C checkpoint saved to Drive.')
    else:
        print('⚠️  No best.pth found — training may have been too short.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

INFO: ProsodyHeads: 50,182 parameters (F0: 25,220, Energy: 24,962)


🚀 Starting System C training (+ prosody heads, λ=0.1) ...
   Init from: System B | Epochs: 100, LR: 0.0001, Batch: 16


/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")
INFO: Downloading model to /root/.local/share/tts/tts_models--en--ljspeech--vits
100%|██████████| 146M/146M [00:02<00:00, 59.2MiB/s]
INFO: Model's license - apache 2.0
INFO: Check https://choosealicense.com/licenses/apache-2.0/ for more info.
INFO: Using model: vits
INFO: Setting up Audio Processor...
INFO:  | sample_rate: 22050
INFO:  | resample: False
INFO:  | num_mels: 80
INFO:  | log_func: np.log10
INFO:  | min_level_db: 0
INFO:  | frame_shift_ms: None
INFO:  | frame_length_ms: None
INFO:  | ref_level_db: None
INFO:  | fft_size: 1024
INFO:  | power: None
INFO:  | preemphasis: 0.0
INFO:  | griffin_lim_iters: None
INFO:  | signal_norm: None
INFO:  | symmetric_norm: None
INFO:  | mel_fmin: 0
INFO:  | mel_fmax: None
INFO:  | pitch_fmin: None
INFO:  | pitch_fmax: None
INFO:  | 

RuntimeError: a Tensor with 16 elements cannot be converted to Scalar

In [ ]:
# ══ 4b. Training Loss Curves ══
#
# Visualize loss convergence for any system that was trained this session.
# Skipped if all checkpoints were restored from Drive (nothing to plot).
#
import matplotlib.pyplot as plt
import pandas as pd

_histories = {}
for name, var in [('A', 'history_a'), ('B', 'history_b'), ('C', 'history_c')]:
    h = globals().get(var)
    if h and len(h) > 0:
        _histories[name] = pd.DataFrame(h)

if _histories:
    loss_keys = ['train_loss_total', 'val_loss_total', 'train_loss_kl',
                 'train_loss_dur', 'train_loss_mel']
    labels    = ['Train Total', 'Val Total', 'KL', 'Duration', 'Mel (monitor)']
    colors    = ['#2c3e50', '#e74c3c', '#3498db', '#2ecc71', '#95a5a6']
    styles    = ['-', '--', ':', ':', ':']

    n_sys = len(_histories)
    fig, axes = plt.subplots(1, n_sys, figsize=(7 * n_sys, 5), squeeze=False)

    for idx, (sys_name, df) in enumerate(_histories.items()):
        ax = axes[0][idx]
        for key, label, color, ls in zip(loss_keys, labels, colors, styles):
            if key in df.columns:
                ax.plot(df['epoch'], df[key], label=label, color=color,
                        linestyle=ls, linewidth=2 if ls == '-' else 1.2)
        ax.set_title(f'System {sys_name} Training Curves', fontsize=13)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()
    print('✅ Training curves plotted.')
else:
    print('ℹ️  No training ran this session (checkpoints restored) — nothing to plot.')

## 5. Inference

Generate evaluation stimuli: **16 canary texts × 4 emotions × 4 systems = 256 samples**

In [ ]:
# ══ 5a. Generate evaluation stimuli for all 4 systems ══
#
# 16 canary texts × 4 emotions × 4 systems = 256 audio samples
#
import os, yaml, torch, shutil, gc

EVAL_DIR = 'data/processed/eval_stimuli'
EVAL_MANIFEST = os.path.join(EVAL_DIR, 'eval_manifest.csv')

# Restore from Drive if available
drive_eval = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'eval_stimuli')
drive_manifest = os.path.join(drive_eval, 'eval_manifest.csv')
if not os.path.isfile(EVAL_MANIFEST) and os.path.isfile(drive_manifest):
    os.makedirs(EVAL_DIR, exist_ok=True)
    shutil.copytree(drive_eval, EVAL_DIR, dirs_exist_ok=True)
    print('Restored eval stimuli from Drive.')

if os.path.isfile(EVAL_MANIFEST):
    import pandas as pd
    n = len(pd.read_csv(EVAL_MANIFEST))
    print(f'✅ Eval manifest exists ({n} entries) — SKIPPING inference.')
else:
    # Pre-flight: verify all checkpoints exist
    _missing = []
    for sys_name in ['A', 'B', 'C']:
        local = f'checkpoints/system_{sys_name.lower()}/best.pth'
        drive = os.path.join(DRIVE_PROJECT, 'checkpoints', f'system_{sys_name.lower()}', 'best.pth')
        if not os.path.isfile(local):
            if os.path.isfile(drive):
                os.makedirs(os.path.dirname(local), exist_ok=True)
                shutil.copy2(drive, local)
                print(f'  Restored System {sys_name} checkpoint from Drive')
            else:
                _missing.append(sys_name)
    if _missing:
        print(f'⚠️  Missing checkpoints for: {_missing} — these systems will be skipped.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.inference.run import run_inference

    infer_cfg = {
        'inference': {
            'use_cuda': torch.cuda.is_available(),
            'systems': ['A0', 'A', 'B', 'C'],
            'output_dir': EVAL_DIR,
            'canary_texts': 'configs/canary_texts.txt',
            'system_a_checkpoint': 'checkpoints/system_a/best.pth',
            'system_b_checkpoint': 'checkpoints/system_b/best.pth',
            'system_c_checkpoint': 'checkpoints/system_c/best.pth',
            'noise_scale': 0.667,
            'length_scale': 1.0,
        }
    }

    print('🚀 Generating evaluation samples for all 4 systems ...')
    results = run_inference(infer_cfg)
    print(f'✅ Inference complete: {results.get("total_files", 0)} files generated')

    # Backup to Drive
    os.makedirs(drive_eval, exist_ok=True)
    shutil.copytree(EVAL_DIR, drive_eval, dirs_exist_ok=True)
    print('Eval stimuli saved to Drive.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6. Evaluation

Run prosody analysis (PRIMARY metric) and SER probe (AUXILIARY metric) on all generated eval stimuli.

- **6a** Prosody analysis — F0 & energy statistics per system × emotion
- **6b** SER probe — Wav2Vec2 emotion classification (auxiliary only)
- **6c** Listening test stimulus pack
- **6d** Save all outputs to Drive

In [ ]:
# ══ 6a. Prosody analysis (PRIMARY metric) ══
import os, yaml, pandas as pd
from IPython.display import display, HTML

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Running prosody evaluation ...')
try:
    from src.evaluation.prosody import run_prosody_evaluation
    prosody_results = run_prosody_evaluation(eval_cfg)
    print('✅ Prosody evaluation complete.\n')

    # Display aggregate statistics table
    if 'agg_stats' in prosody_results:
        print('📊 Aggregate Prosody Stats (System × Emotion):')
        display(prosody_results['agg_stats'])

    # Display causal attribution results
    if 'causal_results' in prosody_results and len(prosody_results['causal_results']) > 0:
        causal_df = prosody_results['causal_results']
        sig_df = causal_df[causal_df.get('significant', False) == True] if 'significant' in causal_df.columns else pd.DataFrame()
        if len(sig_df) > 0:
            print(f'\n🔬 Significant causal effects ({len(sig_df)} found):')
            display(sig_df[['comparison', 'metric', 'emotion', 'mean_diff', 'p_value', 'interpretation']].head(20))
        else:
            print('\nNo statistically significant causal effects found (may need more data).')

    # Display emotion differentiation tests
    if 'diff_tests' in prosody_results:
        diff_df = prosody_results['diff_tests']
        print('\n📈 Emotion Differentiation Tests (Kruskal-Wallis):')
        display(diff_df)

except Exception as e:
    print(f'❌ Prosody analysis failed: {e}')
    import traceback; traceback.print_exc()

In [ ]:
# ══ 6b. SER probe (AUXILIARY metric — non-critical) ══
import yaml
from IPython.display import display

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Running SER probe evaluation ...')
try:
    from src.evaluation.ser_probe import run_ser_evaluation
    ser_results = run_ser_evaluation(eval_cfg)
    agreement = ser_results.get('ser_proxy_agreement', 0)
    n_excluded = ser_results.get('n_excluded', 0)

    print(f'\n✅ SER proxy agreement: {agreement:.2%}')
    print(f'   NOTE: This is an AUXILIARY metric only (Wav2Vec2 proxy).')
    if n_excluded > 0:
        print(f'   Excluded {n_excluded} samples with unmapped emotions (disgust).')

    # Per-system breakdown
    per_system = ser_results.get('per_system', {})
    if per_system:
        print('\n📊 SER Agreement by System:')
        for sys_name in ['A0', 'A', 'B', 'C']:
            if sys_name in per_system:
                print(f'   System {sys_name}: {per_system[sys_name]:.2%}')

    # Per-emotion breakdown
    per_emotion = ser_results.get('per_emotion', {})
    if per_emotion:
        print('\n📊 SER Agreement by Emotion:')
        for emo, acc in per_emotion.items():
            print(f'   {emo:12s}: {acc:.2%}')

except Exception as e:
    print(f'⚠️  SER probe failed (non-critical): {e}')
    print('Continuing without SER results — prosody is the primary metric.')

In [ ]:
# ══ 6c. Listening test stimulus pack ══
import os

EVAL_MANIFEST = 'data/processed/eval_stimuli/eval_manifest.csv'
if os.path.isfile(EVAL_MANIFEST):
    try:
        from src.evaluation.listening_test import create_stimulus_pack
        print('Creating listening test stimulus pack ...')
        stim_summary = create_stimulus_pack(
            manifest_path=EVAL_MANIFEST,
            output_dir='outputs/listening_test',
        )
        print(f'✅ Stimulus pack: {stim_summary}')
    except Exception as e:
        print(f'⚠️  Listening test pack failed (non-critical): {e}')
else:
    print('No eval manifest — run Section 5 first.')

In [ ]:
# ══ 6d. Save ALL outputs to Google Drive ══
import os, shutil

for folder in ['tables', 'figures', 'outputs', 'docs']:
    if os.path.isdir(folder):
        dst = os.path.join(DRIVE_PROJECT, folder)
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f'  ✓ {folder}/ → Drive')
    else:
        print(f'  — {folder}/ not found, skipping')

print('\n✅ All outputs saved to Google Drive.')

## 7. System Comparison — Visualization

Generate publication-quality plots comparing all four systems across prosody metrics and SER results.
This section reads from the evaluation CSVs and produces inline visualizations.

**Key comparisons:**
- F0 and energy differentiation across emotions (do systems B/C produce distinct prosody per emotion?)
- Causal chain: A0 → A → B → C (what does each modification contribute?)
- SER confusion matrices (does the auxiliary classifier agree with intended emotion?)

In [ ]:
# ══ 7a. Generate all evaluation plots ══
import os, yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Generating evaluation plots ...')
try:
    from src.evaluation.plots import generate_all_plots
    generate_all_plots(eval_cfg)
    print('✅ All plots saved to figures/')
except Exception as e:
    print(f'⚠️  Plot generation failed: {e}')
    import traceback; traceback.print_exc()

# List generated figures
if os.path.isdir('figures'):
    figs = sorted(os.listdir('figures'))
    print(f'\n📁 Generated {len(figs)} figures:')
    for f in figs:
        print(f'   {f}')

In [ ]:
# ══ 7b. F0 by System × Emotion — PRIMARY comparison plot ══
#
# This is the key figure: does each system produce increasingly
# differentiated F0 patterns across emotions?
#
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

FIGURES_DIR = 'figures'
TABLES_DIR = 'tables'

# Display pre-generated plot if available
f0_plot = os.path.join(FIGURES_DIR, 'f0_by_system_emotion.png')
if os.path.isfile(f0_plot):
    print('📊 F0 Mean by System × Emotion (pre-generated):')
    display(Image(filename=f0_plot, width=900))
else:
    # Generate inline from prosody CSV
    prosody_csv = os.path.join(TABLES_DIR, 'prosody_analysis.csv')
    if os.path.isfile(prosody_csv):
        df = pd.read_csv(prosody_csv)

        SYSTEM_ORDER = ['A0', 'A', 'B', 'C']
        EMOTION_ORDER = ['neutral', 'angry', 'amused', 'disgust']
        EMOTION_COLORS = {'neutral': '#95a5a6', 'angry': '#e74c3c',
                          'amused': '#f39c12', 'disgust': '#9b59b6'}

        df['system'] = pd.Categorical(df['system'], categories=SYSTEM_ORDER, ordered=True)
        df['emotion'] = pd.Categorical(df['emotion'], categories=EMOTION_ORDER, ordered=True)

        fig, ax = plt.subplots(figsize=(12, 6))
        sns.boxplot(data=df, x='system', y='f0_mean', hue='emotion',
                    palette=EMOTION_COLORS, ax=ax)
        ax.set_title('F0 Mean by System × Emotion', fontsize=14)
        ax.set_xlabel('System')
        ax.set_ylabel('F0 Mean (Hz)')
        ax.legend(title='Emotion', loc='upper right')
        fig.tight_layout()
        plt.show()
    else:
        print('No prosody data available yet. Run Section 6a first.')

In [ ]:
# ══ 7c. Prosody Comparison Grid — 4 metrics at once ══
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Display pre-generated grid plot if available
grid_plot = os.path.join(FIGURES_DIR, 'prosody_comparison_grid.png')
if os.path.isfile(grid_plot):
    print('📊 Prosody Comparison Grid (F0 mean/std, Energy mean/std):')
    display(Image(filename=grid_plot, width=1000))
else:
    prosody_csv = os.path.join(TABLES_DIR, 'prosody_analysis.csv')
    if os.path.isfile(prosody_csv):
        df = pd.read_csv(prosody_csv)
        SYSTEM_ORDER = ['A0', 'A', 'B', 'C']
        EMOTION_COLORS = {'neutral': '#95a5a6', 'angry': '#e74c3c',
                          'amused': '#f39c12', 'disgust': '#9b59b6'}
        df['system'] = pd.Categorical(df['system'], categories=SYSTEM_ORDER, ordered=True)

        metrics = ['f0_mean', 'f0_std', 'energy_mean', 'energy_std']
        titles  = ['F0 Mean (Hz)', 'F0 Std (Hz)', 'Energy Mean', 'Energy Std']

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        for ax, metric, title in zip(axes.flatten(), metrics, titles):
            if metric in df.columns:
                sns.boxplot(data=df, x='system', y=metric, hue='emotion',
                            palette=EMOTION_COLORS, ax=ax)
                ax.set_title(title, fontsize=13)
                ax.set_xlabel('System')
                ax.legend(title='Emotion', fontsize=8)
            else:
                ax.set_visible(False)

        fig.suptitle('Prosody Metrics: System × Emotion Comparison', fontsize=16, y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print('No prosody data available yet.')

In [ ]:
# ══ 7d. Causal Chain: A0 → A → B → C progression ══
#
# Visualize what each system modification contributes.
# Uses pre-generated causal chain plots or builds inline.
#
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image

# Display per-emotion causal chain plots
emotions = ['angry', 'amused', 'disgust', 'neutral']
causal_found = False

for emo in emotions:
    plot_path = os.path.join(FIGURES_DIR, f'causal_chain_{emo}.png')
    if os.path.isfile(plot_path):
        causal_found = True
        print(f'📊 Causal Chain — {emo.title()}:')
        display(Image(filename=plot_path, width=700))

if not causal_found:
    # Build inline from causal attribution CSV
    causal_csv = os.path.join(TABLES_DIR, 'causal_attribution.csv')
    prosody_csv = os.path.join(TABLES_DIR, 'prosody_analysis.csv')

    if os.path.isfile(prosody_csv):
        df = pd.read_csv(prosody_csv)
        SYSTEM_ORDER = ['A0', 'A', 'B', 'C']
        SYSTEM_COLORS = {'A0': '#95a5a6', 'A': '#3498db', 'B': '#e74c3c', 'C': '#2ecc71'}

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        for ax, emo in zip(axes.flatten(), emotions):
            emo_df = df[df['emotion'] == emo]
            means = []
            for sys in SYSTEM_ORDER:
                vals = emo_df[emo_df['system'] == sys]['f0_std'].dropna()
                means.append(vals.mean() if len(vals) > 0 else 0)

            bars = ax.bar(SYSTEM_ORDER, means,
                          color=[SYSTEM_COLORS[s] for s in SYSTEM_ORDER],
                          edgecolor='black', linewidth=1.2)
            for bar, val in zip(bars, means):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(means)*0.01,
                        f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

            # Draw arrows showing change direction
            for i in range(len(SYSTEM_ORDER)-1):
                if means[i+1] > means[i]:
                    ax.annotate('↑', xy=(i+0.5, max(means)*0.95), fontsize=16,
                                ha='center', color='green', fontweight='bold')
                elif means[i+1] < means[i]:
                    ax.annotate('↓', xy=(i+0.5, max(means)*0.95), fontsize=16,
                                ha='center', color='red', fontweight='bold')

            ax.set_title(f'{emo.title()} — F0 Std Progression', fontsize=13)
            ax.set_ylabel('F0 Std (Hz)')

        fig.suptitle('Causal Chain: What Does Each System Add?', fontsize=16)
        fig.tight_layout()
        plt.show()
    else:
        print('No prosody data available yet.')

In [ ]:
# ══ 7e. SER Confusion Matrices — per system ══
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Display pre-generated confusion matrix plot
ser_plot = os.path.join(FIGURES_DIR, 'ser_confusion.png')
if os.path.isfile(ser_plot):
    print('📊 SER Probe Confusion Matrices (auxiliary metric):')
    display(Image(filename=ser_plot, width=1000))
else:
    ser_csv = os.path.join(TABLES_DIR, 'ser_probe_results.csv')
    if os.path.isfile(ser_csv):
        ser_df = pd.read_csv(ser_csv)
        SYSTEM_ORDER = ['A0', 'A', 'B', 'C']
        systems = [s for s in SYSTEM_ORDER if s in ser_df['system'].unique()]

        if systems and 'predicted_mapped' in ser_df.columns:
            fig, axes = plt.subplots(1, len(systems), figsize=(5 * len(systems), 5))
            if len(systems) == 1:
                axes = [axes]

            for ax, sys_name in zip(axes, systems):
                sys_df = ser_df[ser_df['system'] == sys_name]
                emotions = sorted(sys_df['emotion'].unique())
                cm = pd.crosstab(sys_df['emotion'], sys_df['predicted_mapped'],
                                 normalize='index')
                sns.heatmap(cm, annot=True, fmt='.2f', cmap='YlOrRd',
                            ax=ax, vmin=0, vmax=1)
                ax.set_title(f'System {sys_name}')
                ax.set_xlabel('SER Predicted')
                ax.set_ylabel('Intended Emotion')

            fig.suptitle('SER Probe Confusion Matrices (auxiliary metric)', fontsize=14)
            fig.tight_layout()
            plt.show()
        else:
            print('SER results available but no predicted_mapped column.')
    else:
        print('No SER results available — SER probe may have been skipped.')

In [ ]:
# ══ 7f. Quantitative Comparison Summary Table ══
#
# Build a clean summary table comparing all 4 systems across key metrics.
#
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

prosody_csv = os.path.join(TABLES_DIR, 'prosody_analysis.csv')
ser_csv = os.path.join(TABLES_DIR, 'ser_probe_results.csv')

if os.path.isfile(prosody_csv):
    df = pd.read_csv(prosody_csv)
    SYSTEM_ORDER = ['A0', 'A', 'B', 'C']

    # Build summary table
    summary_rows = []
    for sys in SYSTEM_ORDER:
        sys_df = df[df['system'] == sys]
        row = {'System': sys}

        # Overall prosody metrics
        for metric in ['f0_mean', 'f0_std', 'energy_mean', 'energy_std']:
            if metric in sys_df.columns:
                row[f'{metric} (mean±std)'] = f"{sys_df[metric].mean():.1f} ± {sys_df[metric].std():.1f}"

        # Emotion spread: std of per-emotion means (higher = more differentiation)
        if 'f0_mean' in sys_df.columns:
            emo_means = sys_df.groupby('emotion')['f0_mean'].mean()
            row['F0 Emotion Spread'] = f'{emo_means.std():.1f} Hz'

        if 'f0_std' in sys_df.columns:
            emo_stds = sys_df.groupby('emotion')['f0_std'].mean()
            row['F0 Std Spread'] = f'{emo_stds.std():.1f} Hz'

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    # Add SER agreement if available
    if os.path.isfile(ser_csv):
        ser_df = pd.read_csv(ser_csv)
        if 'predicted_mapped' in ser_df.columns:
            for i, sys in enumerate(SYSTEM_ORDER):
                sys_ser = ser_df[ser_df['system'] == sys]
                # Exclude unmapped emotions (disgust)
                mapped = sys_ser[sys_ser['emotion'] != 'disgust']
                if len(mapped) > 0:
                    correct = (mapped['emotion'] == mapped['predicted_mapped']).mean()
                    summary_df.loc[i, 'SER Agreement'] = f'{correct:.1%}'

    print('📊 System Comparison Summary:')
    print('=' * 80)
    display(summary_df.set_index('System'))

    # Interpretation
    print('\n📝 Interpretation Guide:')
    print('  • F0 Emotion Spread: Higher = more F0 differentiation between emotions (GOOD)')
    print('  • F0 Std Spread: Higher = more expressive variation per emotion (GOOD)')
    print('  • SER Agreement: Higher = generated audio classified as intended emotion (GOOD)')
    print()
    print('  Expected trend: A0 ≤ A < B ≤ C (each system should improve over previous)')
else:
    print('No prosody data available. Run sections 5 and 6 first.')

In [ ]:
# ══ 7g. Statistical Significance Heatmap ══
#
# Visualize which system transitions produced statistically significant
# improvements for each emotion × metric combination.
#
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

causal_csv = os.path.join(TABLES_DIR, 'causal_attribution.csv')
if os.path.isfile(causal_csv):
    causal_df = pd.read_csv(causal_csv)

    if 'comparison' in causal_df.columns and 'p_value' in causal_df.columns:
        # Create a heatmap of effect sizes for each comparison × emotion × metric
        comparisons = ['A0→A', 'A→B', 'B→C']
        emotions = ['neutral', 'angry', 'amused', 'disgust']
        metrics = ['f0_mean', 'f0_std', 'energy_mean']

        fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
        if len(metrics) == 1:
            axes = [axes]

        for ax, metric in zip(axes, metrics):
            # Build effect size matrix
            matrix = pd.DataFrame(0.0, index=emotions, columns=comparisons)
            sig_matrix = pd.DataFrame('', index=emotions, columns=comparisons)

            for _, row in causal_df.iterrows():
                if (row.get('metric') == metric and
                    row.get('emotion') in emotions and
                    row.get('comparison') in comparisons):
                    effect = row.get('mean_diff', 0)
                    p_val = row.get('p_value', 1)
                    matrix.loc[row['emotion'], row['comparison']] = effect
                    if p_val < 0.001:
                        sig_matrix.loc[row['emotion'], row['comparison']] = '***'
                    elif p_val < 0.01:
                        sig_matrix.loc[row['emotion'], row['comparison']] = '**'
                    elif p_val < 0.05:
                        sig_matrix.loc[row['emotion'], row['comparison']] = '*'

            # Create annotated heatmap
            annot = matrix.round(2).astype(str) + '\n' + sig_matrix
            vmax = max(abs(matrix.values.max()), abs(matrix.values.min()), 1)

            sns.heatmap(matrix, annot=annot, fmt='', cmap='RdBu_r',
                        center=0, vmin=-vmax, vmax=vmax, ax=ax,
                        linewidths=1, linecolor='white')
            ax.set_title(f'{metric.replace("_", " ").title()}', fontsize=13)
            ax.set_ylabel('Emotion')

        fig.suptitle('Causal Attribution: Effect Sizes (with significance)\n'
                     '* p<0.05  ** p<0.01  *** p<0.001', fontsize=14)
        fig.tight_layout()
        plt.show()
    else:
        print('Causal attribution CSV exists but missing required columns.')
else:
    print('No causal attribution data. Run Section 6a first.')

In [ ]:
# ══ 7h. Audio Samples — Listen to generated speech ══
#
# Play a few sample audio files for manual inspection.
#
import os
import pandas as pd
from IPython.display import Audio, display as ipy_display

EVAL_MANIFEST = 'data/processed/eval_stimuli/eval_manifest.csv'
if os.path.isfile(EVAL_MANIFEST):
    df = pd.read_csv(EVAL_MANIFEST)

    # Select one text, show all system × emotion variants
    sample_text_id = df['text_id'].iloc[0] if 'text_id' in df.columns else 1
    sample = df[df['text_id'] == sample_text_id]

    print(f'🔊 Audio samples for text_id={sample_text_id}:')
    if 'text' in sample.columns:
        print(f'   "{sample["text"].iloc[0]}"')
    print()

    for emotion in ['neutral', 'angry', 'amused']:
        emo_samples = sample[sample['emotion'] == emotion]
        for _, row in emo_samples.iterrows():
            fpath = row.get('file_path', '')
            sys_name = row.get('system', '?')
            if os.path.isfile(fpath):
                print(f'  System {sys_name} — {emotion}:')
                ipy_display(Audio(fpath))
else:
    print('No eval samples available — run Sections 5 & 6 first.')

In [ ]:
# ══ 7i. Final Save — persist figures and tables to Drive ══
import os, shutil

for folder in ['tables', 'figures', 'outputs']:
    if os.path.isdir(folder):
        dst = os.path.join(DRIVE_PROJECT, folder)
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        n = len(os.listdir(folder))
        print(f'  ✓ {folder}/ ({n} files) → Drive')

print('\n✅ All visualizations and results saved to Google Drive.')

## 8. Summary

### Pipeline Steps Completed

| Step | Section | Description | Output |
|------|---------|-------------|--------|
| Data prep | S1 | EmoV-DB downloaded, processed, splits created | `data/manifests/*.csv` |
| System A | S2 | Domain-adapted VITS (no emotion labels) | `checkpoints/system_a/best.pth` |
| System B | S3 | + emotion embedding `nn.Embedding(4, 192)` | `checkpoints/system_b/best.pth` |
| System C | S4 | + prosody heads (F0 + energy, λ=0.1) | `checkpoints/system_c/best.pth` |
| Loss curves | S4b | Training convergence plots (KL, dur, mel) | Inline |
| Inference | S5 | 16 texts × 4 emotions × 4 systems = 256 samples | `data/processed/eval_stimuli/` |
| Prosody | S6a | PRIMARY — F0/energy stats + statistical tests | `tables/prosody_analysis.csv` |
| SER probe | S6b | AUXILIARY — Wav2Vec2 proxy agreement | `tables/ser_probe_results.csv` |
| **Visualization** | **S7** | **System comparison plots + analysis** | **`figures/*.png`** |

### Training Loss Components

| Loss | Participates in Backward? | Purpose |
|------|---------------------------|---------|
| `kl_loss` | ✅ Yes | Flow alignment — primary training signal |
| `duration_loss` | ✅ Yes | Duration predictor (SDP NLL from MAS alignment) |
| `prosody_loss` | ✅ Yes (System C only) | Auxiliary prosody heads (F0 + energy, λ=0.1) |
| `mel_loss` | ❌ No (monitoring only) | Frozen decoder — constant, logged for reference |

### Key Outputs

| File | Description |
|------|-------------|
| `figures/f0_by_system_emotion.png` | F0 Mean boxplot — primary comparison |
| `figures/prosody_comparison_grid.png` | 4-metric grid (F0 mean/std, energy mean/std) |
| `figures/causal_chain_*.png` | Per-emotion causal attribution bar charts |
| `figures/ser_confusion.png` | SER probe confusion matrices per system |
| `tables/prosody_stats_aggregate.csv` | Aggregate prosody statistics |
| `tables/causal_attribution.csv` | Pairwise causal comparisons with p-values |
| `tables/emotion_differentiation_tests.csv` | Kruskal-Wallis test results |

### Expected Results

- **A0 → A**: Minimal change (domain adaptation alone doesn't add emotion)
- **A → B**: Emotion embedding should increase F0/energy differentiation across emotions
- **B → C**: Prosody supervision should further sharpen the difference

**All checkpoints and outputs** persisted to Google Drive at `/content/drive/MyDrive/COMP3065_Project/`

> **Resumability:** Re-run from top after disconnect — completed stages are detected and skipped automatically.
> To retrain from scratch after code changes, set `FORCE_RETRAIN = True` in cell 1c.